# PolarViT baseline — explainability diagnosis (`pipelineLithosXAI.ipynb`)

Sister notebook to `pipeline.ipynb`. That notebook builds the baseline + calibration arm; **this one asks a
different question: *where does the frozen baseline actually look, and on which polarisation?*** — the
detect step of detect→enforce→verify. Everything here runs on the **frozen** checkpoints (no training).

**Method ladder (built over the weekend):** Grad-CAM per branch (this batch) → attention rollout → the fusion
cross-attention maps → Chefer generic relevance (the architecture-matched headline). Grad-CAM is the familiar,
lowest-fidelity comparator (xaicompare2025); Chefer is the method matched to PolarViT's cross-attention.

> **New dependency:** `pip install grad-cam` (PyPI name for `pytorch_grad_cam`) before running Stage 2.

## Stage 1 — stand up the frozen model + pick exemplars

> *Learning pointer.* Cell below is the same validated foundation as `pipeline.ipynb` Stages 0–3, gathered into
> one block so this notebook is self-contained. Nothing new — if a line here puzzles you, the long-form
> explanation lives in the matching `pipeline.ipynb` cell.

In [ ]:
%pip install grad-cam

In [ ]:
import os, sys, math, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from skimage import io
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import vit_l_16, ViT_L_16_Weights

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- paths (identical to pipeline.ipynb cell 6) ------------------------------------------------
PIPELINE_DIR = Path.cwd()                                   # must be code/pipeline
CODE_DIR     = PIPELINE_DIR.parent
LITHOS_CLONE = CODE_DIR / "external" / "lithos"
LITHOS_DATA    = Path(os.environ.get("LITHOS_DATA", r"S:\datasets\lithos"))
LITHOS_WEIGHTS = Path(os.environ.get("LITHOS_WEIGHTS", str(LITHOS_DATA / "final_weights")))
ARCHIVE      = LITHOS_DATA / "archive"
DATASET_ROOT = ARCHIVE / "LITHOS_DATASET_P" / "LITHOS_DATASET"

def fold_csv(fold): return ARCHIVE / f"Fold{fold}_complete_info_allclasses.csv"
def weights(name):  return LITHOS_WEIGHTS / name
def patch_dir(sec_patch): return os.path.join(DATASET_ROOT, os.path.dirname(sec_patch))

def load_patch_images(sec_patch):
    """PPL and XPL images of one patch as (H,W,3) uint8 RGB arrays - the two dual-encoder inputs."""
    d = patch_dir(sec_patch)
    return np.asarray(io.imread(os.path.join(d, "ppl-0.png"))), \
           np.asarray(io.imread(os.path.join(d, "xpl-0.png")))

# --- 29 -> 25 label merge (identical to pipeline.ipynb) ----------------------------------------
_MERGE = {4: 24, 21: 24, 25: 24, 26: 24}
_SURVIVING = sorted(set(range(25 + len(_MERGE))) - set(_MERGE))
RAW_TO_MODEL = {raw: i for i, raw in enumerate(_SURVIVING)}; RAW_TO_MODEL.update(_MERGE)

# --- the backbone's own test-time transform ----------------------------------------------------
preprocess = ViT_L_16_Weights.IMAGENET1K_SWAG_LINEAR_V1.transforms()
to_tensor  = transforms.ToTensor()
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| weights dir exists:", LITHOS_WEIGHTS.exists())

> *Learning pointer.* `strict=False` + the `TIED_ALIAS_KEYS` allow-list is the crux: the checkpoint legitimately
> omits four keys that are **tied aliases** (they share storage with a saved sibling, so the value still arrives).
> Any *other* missing/unexpected key means the wrong file — so we fail loud on those but forgive the four.

In [ ]:
if str(LITHOS_CLONE) not in sys.path:
    sys.path.insert(0, str(LITHOS_CLONE))
from polar_vit_arch import PolarViT, Decoder, Config       # upstream, unmodified

N_CLASSES = 25
TIED_ALIAS_KEYS = {"vit1.class_token", "vit2.class_token",
                   "classification_head.weight", "classification_head.bias"}

def build_polarvit(n_classes, device):
    swag = ViT_L_16_Weights.IMAGENET1K_SWAG_LINEAR_V1
    vit1 = vit_l_16(weights=swag); vit1.heads.head = nn.Linear(vit1.heads.head.in_features, n_classes)
    vit2 = vit_l_16(weights=swag); vit2.heads.head = nn.Linear(vit2.heads.head.in_features, n_classes)
    cfg = Config()
    dec1 = Decoder(cfg.hidden_size, cfg.num_layers, cfg.num_attention_heads)
    dec2 = Decoder(cfg.hidden_size, cfg.num_layers, cfg.num_attention_heads)
    return PolarViT(vit1, vit2, dec1, dec2, vit2.heads.head, use_alpha=True, use_linear=False).to(device)

def load_checkpoint(model, ckpt_name, n_classes=N_CLASSES):
    state = torch.load(weights(ckpt_name), map_location="cpu", weights_only=True)
    state = state.get("state_dict", state) if isinstance(state, dict) else state   # _b wrapped, _a bare
    r = model.load_state_dict(state, strict=False)
    bad = set(r.missing_keys) - TIED_ALIAS_KEYS
    assert not bad, f"unexpected MISSING keys -> wrong checkpoint: {sorted(bad)}"
    assert not r.unexpected_keys, f"unexpected EXTRA keys: {r.unexpected_keys}"
    assert model.classification_head.weight.shape[0] == n_classes, "head dim != 25"
    model.eval(); return model

model = build_polarvit(N_CLASSES, device)                  # prints the authors' "Using alpha weighted..." line
model = load_checkpoint(model, "polarvit_multiclass25_a.pth")   # _a: trained Fold 1 -> explains held-out Fold 2
print("loaded _a | params:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f} M")

> *Learning pointer.* Stage-0 finding made concrete: alpha is **one learned scalar per model**, not per-image. So
> the whole model has a single overall PPL/XPL balance — `sigmoid(alpha)` weights branch 1 (PPL), `1-sigmoid(alpha)`
> weights branch 2 (XPL). This is the model's *own* statement of modality reliance, for free, before any
> attribution. Per-exemplar balance has to come from ablation/Chefer later — alpha can't give it.

In [ ]:
def read_alpha(ckpt_name):
    m = load_checkpoint(model, ckpt_name)                  # reuse the one model object (no 2nd 674M on GPU)
    return torch.sigmoid(m.alpha).item()

for ck in ["polarvit_multiclass25_a.pth", "polarvit_multiclass25_b.pth"]:
    a = read_alpha(ck)
    print(f"{ck:32s}  sigmoid(alpha) = {a:.3f}  ->  PPL {a:.0%} / XPL {1-a:.0%}")

# leave the model holding _a for the rest of the notebook (Fold-2 exemplars are _a's held-out set)
model = load_checkpoint(model, "polarvit_multiclass25_a.pth")

> *Learning pointer.* We diagnose on grains the model gets **right** — a saliency map over a wrong prediction is
> ambiguous (is the map wrong, or the model?). Fold 2 is `_a`'s held-out set, so these are honest test-set grains.
> Monocrystalline (0) and Polycrystalline (2) are the two quartz-texture classes your Wk-1 colour-flip PoC swapped
> between — they're the exemplars that carry the compound-signal story.

In [ ]:
f2 = pd.read_csv(PIPELINE_DIR / "cache" / "fold2_labels.csv").reset_index(drop=True)
preds = np.load(PIPELINE_DIR / "cache" / "preds_mc25_a_fold2.npz")["y_pred"]
assert len(preds) == len(f2), "preds/labels misaligned"
f2 = f2.assign(y_pred=preds)
correct = f2[f2["y_pred"] == f2["model_class"]]

# key classes for the diagnosis figure: the quartz-texture trio (compound-signal story) + 3 contrasts
KEY_CLASSES = {0: "Monocrystalline", 2: "Polycrystalline", 4: "Quartz",
               1: "Rock fragment",   3: "Feldspar",        6: "Foraminifer"}
N_PER = 2

rng = np.random.default_rng(SEED)
picks = []
for cls, name in KEY_CLASSES.items():
    pool = correct[correct["model_class"] == cls]
    take = pool.iloc[rng.choice(len(pool), size=min(N_PER, len(pool)), replace=False)]
    for _, row in take.iterrows():
        picks.append({"sec_patch": row["sec_patch"], "model_class": int(cls), "name": name})
exemplars = pd.DataFrame(picks)
print(f"{len(exemplars)} exemplars across {exemplars['name'].nunique()} classes")
print(exemplars[["name", "sec_patch"]].to_string(index=False))

> *Learning pointer.* Two independent confirmations that the wiring is `data1=PPL->vit1`, `data2=XPL->vit2`:
> (1) the loader reads `ppl-0.png` into data1; (2) **swapping the inputs changes the prediction** — proof the
> trained branches are *not* interchangeable (architecturally identical, but each learned modality-specific
> weights). Your belief was right; this turns it into a citable fact for M2.

In [ ]:
import torch.nn.functional as F

def load_pair(sec_patch):
    ppl, xpl = load_patch_images(sec_patch)
    d1 = preprocess(to_tensor(ppl)).unsqueeze(0).to(device)   # PPL -> vit1 (x1)
    d2 = preprocess(to_tensor(xpl)).unsqueeze(0).to(device)   # XPL -> vit2 (x2)
    return ppl, xpl, d1, d2

rows = []
for _, ex in exemplars.iterrows():
    _, _, d1, d2 = load_pair(ex["sec_patch"])
    c = ex["model_class"]
    with torch.no_grad():
        lo, ls = model(d1, d2), model(d2, d1)                 # normal (PPL,XPL) vs swapped (XPL,PPL)
    rows.append({"name": ex["name"], "true": c,
                 "pred_norm": lo.argmax(1).item(), "pred_swap": ls.argmax(1).item(),
                 "p_true_norm": F.softmax(lo, 1)[0, c].item(),
                 "p_true_swap": F.softmax(ls, 1)[0, c].item()})
sw = pd.DataFrame(rows)
sw["flipped"] = sw["pred_norm"] != sw["pred_swap"]

print(f"argmax flipped on PPL<->XPL swap : {sw['flipped'].sum()}/{len(sw)}")
print(f"mean |delta p(true)| on swap     : {(sw['p_true_norm']-sw['p_true_swap']).abs().mean():.3f}"
      f"   (~0 => branches near-interchangeable; large => modality-specific)")
print(sw[["name","true","pred_norm","pred_swap","p_true_norm","p_true_swap","flipped"]].round(3).to_string(index=False))


## Stage 2 — Grad-CAM per branch (the first heatmaps)

Grad-CAM on a ViT: there are no conv feature maps, so we target the **last transformer block's tokens** and
reshape the 196 patch tokens back to a 14x14 grid (`reshape_transform`). We run it **once per branch** — vit1 on
PPL, vit2 on XPL — to see where each polarisation drives the decision. This is the familiar, quick comparator;
xaicompare2025 measured Grad-CAM as *lowest fidelity*, which is exactly why the ladder doesn't stop here.

*No module surgery:* Grad-CAM reads **activation gradients** at the target layer, which flow fine even though the
backbones are frozen (frozen params still pass activation gradients through; the decoders/alpha/head are trainable).

> *Learning pointer.* `GradCAM` calls `model(input_tensor)` with a **single** input, but PolarViT needs **two**.
> The `BranchWrapper` fixes the other modality and varies the branch under study, so Grad-CAM sees a one-input
> model while the real forward stays intact. The target layer (`encoder.layers[-1].ln_1`) still lives on the real
> `model`, so the hook fires correctly.
>
> **Version note:** recent `grad-cam` dropped the `use_cuda` arg (device follows the model). If your installed
> version complains, adjust to match. `cam(...)` returns a (1,H,W) array in [0,1].

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def reshape_transform(tensor, h=14, w=14):
    # torchvision ViT-L/16 @224: (B, 197, 1024) -> drop CLS -> (B,14,14,1024) -> (B,1024,14,14)
    x = tensor[:, 1:, :].reshape(tensor.size(0), h, w, tensor.size(2))
    return x.permute(0, 3, 1, 2)

class BranchWrapper(nn.Module):
    """Expose PolarViT as a single-input model for Grad-CAM: vary `branch`, hold the other modality fixed."""
    def __init__(self, model, other, branch):   # branch=1 -> vary PPL(x1); branch=2 -> vary XPL(x2)
        super().__init__(); self.model = model; self.other = other; self.branch = branch
    def forward(self, x):
        return self.model(x, self.other) if self.branch == 1 else self.model(self.other, x)

def denorm(t):   # model-input tensor (1,3,224,224) -> viewable HWC [0,1], exactly what the model saw
    return (t.squeeze(0).cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

def gradcam_branch(sec_patch, branch, target_class):
    ppl, xpl, d1, d2 = load_pair(sec_patch)
    if branch == 1:   inp, other, layer = d1, d2, model.vit1.encoder.layers[-1].ln_1
    else:             inp, other, layer = d2, d1, model.vit2.encoder.layers[-1].ln_1
    wrapper = BranchWrapper(model, other, branch)
    with GradCAM(model=wrapper, target_layers=[layer], reshape_transform=reshape_transform) as cam:
        gmap = cam(input_tensor=inp, targets=[ClassifierOutputTarget(target_class)])[0]
    return denorm(inp), gmap

# render the exemplars: rows = exemplars, cols = [PPL raw+CAM, XPL raw+CAM]
rows = exemplars.head(6)   # keep the first figure readable; bump later
fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
for i, (_, ex) in enumerate(rows.iterrows()):
    cls = ex["model_class"]
    for j, branch in enumerate([1, 2]):
        base, gmap = gradcam_branch(ex["sec_patch"], branch, cls)
        overlay = show_cam_on_image(base, gmap, use_rgb=True)
        ax[i, 2*j].imshow(base);     ax[i, 2*j].set_title(f"{ex['name']} - {'PPL' if branch==1 else 'XPL'}", fontsize=8)
        ax[i, 2*j+1].imshow(overlay); ax[i, 2*j+1].set_title("Grad-CAM", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Stage 3a — Fusion cross-attention maps (no surgery)

PolarViT fuses the branches with two cross-attention decoders. The authors' `MultiHeadAttention` returns its
attention matrix directly, so a forward hook captures it — no module swap needed. We read the **CLS token's
cross-attention**: for `decoder_1`, PPL's CLS token attends over XPL's patch tokens (→ a map on the **XPL** image);
for `decoder_2`, XPL's CLS attends over PPL's patches (→ a map on the **PPL** image). This answers a question
Grad-CAM cannot: *when the model fuses the two polarisations, which regions of the other modality does each branch
actually consult?*

>Learning pointer. decoder_1(features1, features2) runs PPL (features1) as the query and XPL (features2) as key/value — so its cross-attention is PPL looking at XPL. Token 0 is the CLS (prepended before the encoder), tokens 1..196 are the 14×14 patch grid. Taking the CLS row over the patch tokens, averaged over the 4 heads and the 2 decoder layers, gives one interpretable 14×14 map per direction. No gradients, no surgery — pure forward.

In [ ]:
from skimage.transform import resize

def cls_cross_map(scores):
    """scores (B, heads, 197, 197) -> 224x224 map of the CLS token's cross-attention over the OTHER branch's
    196 patch tokens (head-averaged, upsampled)."""
    a = scores[0].mean(0)[0, 1:]                     # head-avg, CLS row, drop the CLS->CLS entry -> (196,)
    a = a.reshape(14, 14).cpu().numpy()
    a = resize(a, (224, 224), order=1, mode="reflect", anti_aliasing=True)
    return (a - a.min()) / (a.ptp() + 1e-8)          # normalise to [0,1] for display

def fusion_maps(sec_patch):
    store = {}
    def cap(name):
        def h(m, i, o): store[name] = o[1].detach()  # o = (output, scores); scores = o[1]
        return h
    hooks  = [model.decoder_1.layers[l].dec_enc_attn.register_forward_hook(cap(f"d1_{l}"))
              for l in range(len(model.decoder_1.layers))]
    hooks += [model.decoder_2.layers[l].dec_enc_attn.register_forward_hook(cap(f"d2_{l}"))
              for l in range(len(model.decoder_2.layers))]
    _, _, d1, d2 = load_pair(sec_patch)
    with torch.no_grad():
        model(d1, d2)
    for h in hooks: h.remove()
    n = len(model.decoder_1.layers)
    ppl_map = np.mean([cls_cross_map(store[f"d2_{l}"]) for l in range(n)], axis=0)  # XPL-CLS -> PPL regions
    xpl_map = np.mean([cls_cross_map(store[f"d1_{l}"]) for l in range(n)], axis=0)  # PPL-CLS -> XPL regions
    return ppl_map, xpl_map

rows = exemplars.head(6)
fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
for i, (_, ex) in enumerate(rows.iterrows()):
    ppl, xpl, d1, d2 = load_pair(ex["sec_patch"])
    ppl_map, xpl_map = fusion_maps(ex["sec_patch"])
    base_ppl, base_xpl = denorm(d1), denorm(d2)
    ax[i, 0].imshow(base_ppl); ax[i, 0].set_title(f"{ex['name']} - PPL", fontsize=8)
    ax[i, 1].imshow(base_ppl); ax[i, 1].imshow(ppl_map, cmap="jet", alpha=0.5)
    ax[i, 1].set_title("XPL-CLS -> PPL", fontsize=8)
    ax[i, 2].imshow(base_xpl); ax[i, 2].set_title(f"{ex['name']} - XPL", fontsize=8)
    ax[i, 3].imshow(base_xpl); ax[i, 3].imshow(xpl_map, cmap="jet", alpha=0.5)
    ax[i, 3].set_title("PPL-CLS -> XPL", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Stage 3b — Backbone attention + rollout (parity-checked)

Rollout needs each ViT block's self-attention, which torchvision computes with `need_weights=False` (never
materialised). Rather than reimplement attention, we inject `need_weights=True` with a **forward-pre-hook** and
capture the (head-averaged) weights with a forward hook. Because that switches off the fused kernel, we first
**parity-check** that logits are unchanged (same maths, different code path) before trusting anything downstream.

>Learning pointer. A forward_pre_hook(..., with_kwargs=True) can rewrite the call's keyword args before the module runs — here forcing need_weights=True. The matching forward_hook then grabs the returned weights. This is the light-touch alternative to a module swap; the parity assert is what earns the right to use it.

In [ ]:
BACKBONE_ATTN = {}

def _prehook(module, args, kwargs):
    kwargs = dict(kwargs); kwargs["need_weights"] = True; kwargs["average_attn_weights"] = True
    return args, kwargs

def _cap(name):
    def h(m, i, o): BACKBONE_ATTN[name] = o[1].detach()   # (B, L, L) head-averaged
    return h

def attach_backbone_hooks(vit, tag):
    hs = []
    for li, blk in enumerate(vit.encoder.layers):
        mha = blk.self_attention
        hs.append(mha.register_forward_pre_hook(_prehook, with_kwargs=True))
        hs.append(mha.register_forward_hook(_cap(f"{tag}_{li}")))
    return hs

sp = exemplars.iloc[0]["sec_patch"]
_, _, d1, d2 = load_pair(sp)
with torch.no_grad():
    logits_clean = model(d1, d2)

hooks  = attach_backbone_hooks(model.vit1, "v1")
hooks += attach_backbone_hooks(model.vit2, "v2")
with torch.no_grad():
    logits_hooked = model(d1, d2)
for h in hooks: h.remove()

delta = (logits_clean - logits_hooked).abs().max().item()
print(f"parity check: max|logit delta| with hooks = {delta:.2e}   {'OK (<1e-3)' if delta < 1e-3 else 'FAIL - do not trust rollout'}")
print(f"captured {sum(k.startswith('v1') for k in BACKBONE_ATTN)} vit1 layers, "
      f"{sum(k.startswith('v2') for k in BACKBONE_ATTN)} vit2 layers | shape {tuple(next(iter(BACKBONE_ATTN.values())).shape)}")

>Learning pointer. Rollout multiplies each layer's attention (plus an identity for the residual, row-normalised) to approximate how information flows from input patches to the CLS token. It's the cheap attention baseline — better than a single layer, weaker than Chefer (no class-specificity, no cross-attention). Reading it next to the Grad-CAM and fusion maps is the point: agreement is reassuring, disagreement is a question the deletion metrics (Wk 5) will settle.



In [ ]:
def rollout(attn_layers):
    """attn_layers: list of (L, L) head-averaged matrices -> CLS-row over patches as a 14x14 map."""
    R = np.eye(attn_layers[0].shape[-1])
    for A in attn_layers:
        A = A + np.eye(A.shape[-1]); A = A / A.sum(-1, keepdims=True)
        R = A @ R
    cls = R[0, 1:]                                    # CLS attention over the 196 patches
    m = cls.reshape(14, 14)
    m = resize(m, (224, 224), order=1, mode="reflect", anti_aliasing=True)
    return (m - m.min()) / (m.ptp() + 1e-8)

def branch_rollout(sec_patch, tag, vit):
    hooks = attach_backbone_hooks(vit, tag)
    _, _, d1, d2 = load_pair(sec_patch)
    with torch.no_grad():
        model(d1, d2)
    for h in hooks: h.remove()
    n = len(vit.encoder.layers)
    layers = [BACKBONE_ATTN[f"{tag}_{li}"][0].cpu().numpy() for li in range(n)]
    return rollout(layers)

rows = exemplars.head(6)
fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
for i, (_, ex) in enumerate(rows.iterrows()):
    ppl, xpl, d1, d2 = load_pair(ex["sec_patch"])
    base_ppl, base_xpl = denorm(d1), denorm(d2)
    r_ppl = branch_rollout(ex["sec_patch"], "v1", model.vit1)
    r_xpl = branch_rollout(ex["sec_patch"], "v2", model.vit2)
    ax[i, 0].imshow(base_ppl); ax[i, 0].set_title(f"{ex['name']} - PPL", fontsize=8)
    ax[i, 1].imshow(base_ppl); ax[i, 1].imshow(r_ppl, cmap="jet", alpha=0.5); ax[i, 1].set_title("rollout PPL", fontsize=8)
    ax[i, 2].imshow(base_xpl); ax[i, 2].set_title(f"{ex['name']} - XPL", fontsize=8)
    ax[i, 3].imshow(base_xpl); ax[i, 3].imshow(r_xpl, cmap="jet", alpha=0.5); ax[i, 3].set_title("rollout XPL", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()


## Stage 4 — Chefer generic relevance (class-specific)

Rollout shows where attention *flows*; it can't say *why this class*. Chefer's generic method fixes that by weighting
each attention matrix by its gradient w.r.t. the target logit before rolling up: `Ā = mean_heads((∇A ⊙ A)⁺)`. We apply
it in two places — the **cross-attention decoders** (class-specific *cross-modal* relevance: which regions of the
other polarisation the model consulted *for this class*) and the **backbone self-attention** (class-specific
*within-image* relevance). The cross-attention part needs no surgery; the backbone part needs the module swap and is
gated by a parity check.

>Learning pointer. The decoders' custom attention returns the on-graph scores tensor (the one that computes the output), so scores.retain_grad() + one backward() from the target logit gives a real scores.grad. That's the whole trick: (grad ⊙ scores) keeps only the attention the target class actually used, .clamp(min=0) drops the parts that argue against it, head-average, take the CLS row. Compare this to the Batch-2 fusion map (same attention, no gradient) — the difference you see is the class-specificity Chefer adds.

In [ ]:
def avg_heads_grad(A, G):
    """(1,heads,T,T) attention + gradient -> (T,T) class-specific, positive, head-averaged (Chefer's core op)."""
    return (G * A).clamp(min=0).mean(dim=1)[0]

def chefer_cross(sec_patch, target_class):
    store = {}
    def cap(name):
        def h(m, i, o):
            s = o[1]; s.retain_grad(); store[name] = s      # o=(output, scores); keep scores ON the graph
        return h
    n = len(model.decoder_1.layers)
    hooks  = [model.decoder_1.layers[l].dec_enc_attn.register_forward_hook(cap(f"d1_{l}")) for l in range(n)]
    hooks += [model.decoder_2.layers[l].dec_enc_attn.register_forward_hook(cap(f"d2_{l}")) for l in range(n)]

    _, _, d1, d2 = load_pair(sec_patch)
    model.zero_grad(set_to_none=True)
    logits = model(d1, d2)                                   # grad ENABLED (no torch.no_grad here)
    logits[0, target_class].backward()
    for h in hooks: h.remove()

    def cross_map(tag):
        m = np.mean([avg_heads_grad(store[f"{tag}_{l}"], store[f"{tag}_{l}"].grad)[0, 1:]
                     .reshape(14, 14).detach().cpu().numpy() for l in range(n)], axis=0)
        m = resize(m, (224, 224), order=1, mode="reflect", anti_aliasing=True)
        return (m - m.min()) / (m.ptp() + 1e-8)
    return cross_map("d2"), cross_map("d1")                  # (PPL-image map, XPL-image map)

rows = exemplars.head(6)
fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
for i, (_, ex) in enumerate(rows.iterrows()):
    ppl, xpl, d1, d2 = load_pair(ex["sec_patch"])
    ppl_map, xpl_map = chefer_cross(ex["sec_patch"], ex["model_class"])
    bppl, bxpl = denorm(d1), denorm(d2)
    ax[i, 0].imshow(bppl); ax[i, 0].set_title(f"{ex['name']} - PPL", fontsize=8)
    ax[i, 1].imshow(bppl); ax[i, 1].imshow(ppl_map, cmap="jet", alpha=0.5); ax[i, 1].set_title("Chefer XPL->PPL", fontsize=8)
    ax[i, 2].imshow(bxpl); ax[i, 2].set_title(f"{ex['name']} - XPL", fontsize=8)
    ax[i, 3].imshow(bxpl); ax[i, 3].imshow(xpl_map, cmap="jet", alpha=0.5); ax[i, 3].set_title("Chefer PPL->XPL", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

>Learning pointer. α is global and the maps are suggestive; ablation is the closest thing to ground truth — blank one branch (fill it with its own mean = an uninformative grey) and measure how far p(true) falls. A big drop when PPL is removed ⇒ the model needed PPL for that grain; likewise XPL. This is the number the Chefer/α modality story must be consistent with — a built-in sanity check, no attribution required.

In [ ]:
def modality_ablation(sec_patch, target_class):
    _, _, d1, d2 = load_pair(sec_patch)
    m1 = d1.mean(dim=(2, 3), keepdim=True).expand_as(d1)     # neutral (mean) fill = modality removed
    m2 = d2.mean(dim=(2, 3), keepdim=True).expand_as(d2)
    with torch.no_grad():
        p_full  = F.softmax(model(d1, d2), 1)[0, target_class].item()
        p_no_ppl = F.softmax(model(m1, d2), 1)[0, target_class].item()
        p_no_xpl = F.softmax(model(d1, m2), 1)[0, target_class].item()
    return p_full, p_full - p_no_ppl, p_full - p_no_xpl       # full, drop if PPL removed, drop if XPL removed

rows = []
for _, ex in exemplars.iterrows():
    pf, dp, dx = modality_ablation(ex["sec_patch"], ex["model_class"])
    rows.append({"name": ex["name"], "p_full": pf, "drop_noPPL": dp, "drop_noXPL": dx,
                 "leans": "PPL" if dp > dx else "XPL"})
abl = pd.DataFrame(rows)
print("per-exemplar modality reliance (bigger drop = more needed):")
print(abl.round(3).to_string(index=False))
print(f"\nmean drop removing PPL: {abl['drop_noPPL'].mean():.3f} | removing XPL: {abl['drop_noXPL'].mean():.3f}")

## Stage 4b — Within-branch Chefer via module swap (HIGH-RISK, parity-gated)

The backbone self-attention gradients aren't exposed by torchvision (`need_weights=False` returns a head-averaged
side-copy off the gradient path). To get them we **swap** each block's attention for an explicit, numerically-identical
implementation that keeps the on-graph per-head attention. We **parity-check first** (patched logits == original to
< 1e-3) and only trust the within-branch maps if it passes. If parity FAILS, skip this cell — the cross-attention
Chefer (Cell 15) is the validated headline; the backbone maps are an extension, not a dependency.

>Learning pointer. explicit_mha re-does exactly what nn.MultiheadAttention does — pack qkv from in_proj_weight, scale by 1/√head_dim, softmax, out-proj — but keeps attn as a live tensor we can retain_grad(). If the parity assert passes, the swap changed observability, not computation. Then within-branch relevance is Chefer's rule rolled over the 24 blocks: R = R + Ā·R, CLS row → 14×14.

In [ ]:
import types

def explicit_mha(mha, x):
    """Numerically-identical replacement for torchvision's self_attention that exposes on-graph per-head attn."""
    B, T, E = x.shape; H = mha.num_heads; dh = E // H
    qkv = F.linear(x, mha.in_proj_weight, mha.in_proj_bias)
    q, k, v = qkv.chunk(3, dim=-1)
    q = q.view(B, T, H, dh).transpose(1, 2)
    k = k.view(B, T, H, dh).transpose(1, 2)
    v = v.view(B, T, H, dh).transpose(1, 2)
    attn = (q @ k.transpose(-2, -1)) / (dh ** 0.5)
    attn = attn.softmax(dim=-1)
    if attn.requires_grad:        # FIX: guard — the parity forward runs under no_grad (attn has no grad there)
        attn.retain_grad()
    out = (attn @ v).transpose(1, 2).reshape(B, T, E)
    return mha.out_proj(out), attn

def _patched_forward(self, input):
    x = self.ln_1(input)
    x, attn = explicit_mha(self.self_attention, x)
    self._chefer_attn = attn                                  # stash for backward
    x = self.dropout(x); x = x + input
    y = self.ln_2(x); y = self.mlp(y)
    return x + y

def swap_in(vit):
    for blk in vit.encoder.layers:
        blk.forward = types.MethodType(_patched_forward, blk)
def swap_out(vit):
    for blk in vit.encoder.layers:
        del blk.forward                                       # restore the class method

# --- parity check -------------------------------------------------------------------------------
sp = exemplars.iloc[0]["sec_patch"]
_, _, d1, d2 = load_pair(sp)
with torch.no_grad():
    clean = model(d1, d2)
swap_in(model.vit1); swap_in(model.vit2)
with torch.no_grad():
    swapped = model(d1, d2)
delta = (clean - swapped).abs().max().item()
print(f"swap parity: max|logit delta| = {delta:.2e}  {'OK' if delta < 1e-3 else 'FAIL - skip within-branch maps'}")

# --- within-branch Chefer (only trust if parity passed) -----------------------------------------
def chefer_branch(sec_patch, vit, target_class):
    _, _, d1, d2 = load_pair(sec_patch)
    d1.requires_grad_(True); d2.requires_grad_(True)   # FIX: backbone is FROZEN, so grad must enter via the input
    model.zero_grad(set_to_none=True)
    logits = model(d1, d2)
    logits[0, target_class].backward()
    blocks = list(vit.encoder.layers)
    R = torch.eye(blocks[0]._chefer_attn.shape[-1], device=d1.device)
    for blk in blocks:
        A = blk._chefer_attn
        cam = avg_heads_grad(A, A.grad)
        R = R + cam @ R
    m = R[0, 1:].reshape(14, 14).detach().cpu().numpy()
    m = resize(m, (224, 224), order=1, mode="reflect", anti_aliasing=True)
    return (m - m.min()) / (m.ptp() + 1e-8)

if delta < 1e-3:
    rows = exemplars.head(6)
    fig, ax = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
    for i, (_, ex) in enumerate(rows.iterrows()):
        ppl, xpl, d1, d2 = load_pair(ex["sec_patch"])
        r1 = chefer_branch(ex["sec_patch"], model.vit1, ex["model_class"])
        r2 = chefer_branch(ex["sec_patch"], model.vit2, ex["model_class"])
        bppl, bxpl = denorm(d1), denorm(d2)
        ax[i, 0].imshow(bppl); ax[i, 0].set_title(f"{ex['name']} - PPL", fontsize=8)
        ax[i, 1].imshow(bppl); ax[i, 1].imshow(r1, cmap="jet", alpha=0.5); ax[i, 1].set_title("Chefer PPL (within)", fontsize=8)
        ax[i, 2].imshow(bxpl); ax[i, 2].set_title(f"{ex['name']} - XPL", fontsize=8)
        ax[i, 3].imshow(bxpl); ax[i, 3].imshow(r2, cmap="jet", alpha=0.5); ax[i, 3].set_title("Chefer XPL (within)", fontsize=8)
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.show()

swap_out(model.vit1); swap_out(model.vit2)                   # ALWAYS restore the model to pristine
print("model restored to pristine forward")

>Learning pointer. Cell 16 read 6 exemplars; a single grain's ablation is noisy (the mean-fill "removed" image isn't perfectly neutral). Averaging over ~30 correctly-classified grains per class turns that into a stable per-class estimate. Sorting by drop_noXPL − drop_noPPL lines the taxonomy up from PPL-reliant (shape/texture minerals) to XPL-reliant (interference-colour minerals) — the latter are the calibration targets.

In [ ]:
name_of = f2.groupby("model_class")["multiclass_name"].agg(lambda x: x.mode().iat[0]).to_dict()
name_of[24] = "Other (merged)"

N_PER_CLASS = 30
rng = np.random.default_rng(SEED)
recs = []
for cls in range(25):
    pool = correct[correct["model_class"] == cls]
    if len(pool) == 0:
        continue
    take = pool.iloc[rng.choice(len(pool), size=min(N_PER_CLASS, len(pool)), replace=False)]
    dps, dxs = [], []
    for sp in take["sec_patch"]:
        _, dp, dx = modality_ablation(sp, cls)
        dps.append(dp); dxs.append(dx)
    recs.append({"class": cls, "name": name_of.get(cls, str(cls)), "n": len(take),
                 "drop_noPPL": float(np.mean(dps)), "drop_noXPL": float(np.mean(dxs))})
    print(f"  class {cls:2d} {name_of.get(cls,''):16s} n={len(take):2d} "
          f"PPL{np.mean(dps):+.2f} XPL{np.mean(dxs):+.2f}", flush=True)

rel25 = pd.DataFrame(recs)
rel25["xpl_reliance"] = rel25.drop_noXPL - rel25.drop_noPPL     # >0 => XPL(colour)-dominant
rel25["leans"] = np.where(rel25.xpl_reliance > 0, "XPL", "PPL")
rel25 = rel25.sort_values("xpl_reliance").reset_index(drop=True)
print("\n", rel25[["name","n","drop_noPPL","drop_noXPL","leans"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 7))
y = np.arange(len(rel25)); h = 0.4
ax.barh(y - h/2, rel25.drop_noPPL, h, label="drop if PPL removed", color="tab:orange")
ax.barh(y + h/2, rel25.drop_noXPL, h, label="drop if XPL removed", color="tab:blue")
ax.set_yticks(y); ax.set_yticklabels(rel25["name"], fontsize=8)
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("mean drop in p(true class) when a modality is removed")
ax.set_title("Per-class modality reliance (25 classes)\nXPL-leaning (blue > orange) = interference-colour = calibration targets")
ax.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

# save the table — it's a Results artifact (not a throwaway)
rel25.to_csv(PIPELINE_DIR / "cache" / "modality_reliance_25class.csv", index=False)
print("saved -> cache/modality_reliance_25class.csv")

In [ ]:
# ── Combined figure: all 5 methods × the colour-flip exemplars (the fidelity ladder) ────────────
# Learning pointer: nothing new is computed — this cell just CALLS the five method functions the
# notebook already validated (each parity-gated where it needed to be) and lays their outputs on one
# grid so the ladder is visible in a single glance. Order matters: the four no-surgery methods run
# on the pristine model FIRST; the module swap wraps ONLY the within-branch pass, then restores —
# same discipline as Stage 4b (never leave the model swapped).

FLIP_CLASSES = [0, 2]     # Monocrystalline, Polycrystalline — the Wk-1 colour-flip pair
N_PER_CLASS  = 1          # 1 -> 4 columns (readable); 2 -> all four flip exemplars, 8 columns

flips = (exemplars[exemplars["model_class"].isin(FLIP_CLASSES)]
         .groupby("model_class", group_keys=False).head(N_PER_CLASS))

# --- pass 1: pristine model — raw bases + the four no-surgery methods ---------------------------
P = {}                                                     # P[sec_patch] = dict of panels
for _, ex in flips.iterrows():
    sp, cls = ex["sec_patch"], ex["model_class"]
    _, _, d1, d2 = load_pair(sp)
    base_p, base_x = denorm(d1), denorm(d2)                # exactly what the model saw
    _, g_p = gradcam_branch(sp, 1, cls)
    _, g_x = gradcam_branch(sp, 2, cls)
    f_p, f_x = fusion_maps(sp)                             # XPL-CLS->PPL, PPL-CLS->XPL
    r_p = branch_rollout(sp, "v1", model.vit1)
    r_x = branch_rollout(sp, "v2", model.vit2)
    c_p, c_x = chefer_cross(sp, cls)
    P[sp] = {"name": ex["name"], "base": (base_p, base_x),
             "Grad-CAM": (g_p, g_x), "cross-attn": (f_p, f_x),
             "rollout": (r_p, r_x), "Chefer cross": (c_p, c_x)}

# --- pass 2: module swap ONLY for within-branch Chefer, then restore ----------------------------
swap_in(model.vit1); swap_in(model.vit2)
for _, ex in flips.iterrows():
    P[ex["sec_patch"]]["Chefer branch"] = (
        chefer_branch(ex["sec_patch"], model.vit1, ex["model_class"]),
        chefer_branch(ex["sec_patch"], model.vit2, ex["model_class"]))
swap_out(model.vit1); swap_out(model.vit2)                 # ALWAYS restore to pristine

# --- the grid: rows = ladder (low -> high fidelity), cols = exemplar × {PPL, XPL} ---------------
ROWS = ["raw", "Grad-CAM", "cross-attn", "rollout", "Chefer cross", "Chefer branch"]
cols = [(sp, m) for sp in P for m in (0, 1)]               # m: 0 = PPL, 1 = XPL
fig, ax = plt.subplots(len(ROWS), len(cols),
                       figsize=(2.3 * len(cols), 2.3 * len(ROWS)), squeeze=False)
for j, (sp, m) in enumerate(cols):
    base = P[sp]["base"][m]
    ax[0, j].imshow(base)
    ax[0, j].set_title(f"{P[sp]['name']}\n{'PPL' if m == 0 else 'XPL'}", fontsize=8)
    for i, meth in enumerate(ROWS[1:], start=1):
        ax[i, j].imshow(base)
        ax[i, j].imshow(P[sp][meth][m], cmap="jet", alpha=0.5)   # house overlay style
for i, meth in enumerate(ROWS):
    ax[i, 0].set_ylabel(meth, fontsize=9)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle("The fidelity ladder on the colour-flip pair — higher-fidelity methods land on grains",
             fontsize=11, y=0.995)
plt.tight_layout(); plt.show()

## Stage 4c — IROF: the fidelity ladder, quantified

The cell-28 grid says the ladder's ordering *looks* right; IROF (Rieger & Hansen 2020) scores it:
delete superpixels most-important-first per method and measure how fast p(true class) collapses
(area over the curve, AOC — higher = more faithful map). Random deletion = the null rung. Run per
branch, it independently re-tests the XPL-dominance finding from the Stage-4b ablation.

In [ ]:
# ── Stage 4c · Step 1: IROF machinery — superpixel deletion curves, parity-anchored ─────────────
# [prov: lit | Rieger & Hansen 2020, arXiv:2003.08747 (IROF) | impl ours: per-branch deletion for
#  the two-stream model; fill = the input's global per-channel mean so FULL deletion == Stage-4b's
#  mean-fill ablation exactly (asserted below)]
# Segment the input into superpixels, delete them most-important-first per an attribution map, and
# watch p(class) fall: a faithful map front-loads the damage -> steep curve -> high area over the
# curve (AOC). Superpixels (not pixels) keep perturbed images natural-ish and bound the pass count.
from skimage.segmentation import slic

N_SEGMENTS   = 60    # requested superpixels per image — slic MERGES below this (connectivity)
MIN_SEGMENTS = 30    # floor of ACTUAL segments (must exceed N_STEPS); escalate the request if under
N_STEPS      = 20    # deletion steps -> 21 model evaluations per curve (incl. the pristine one)
SUB_BATCH    = 8     # forward sub-batch; model + batch-8 ViT-L inference fits the 11 GB card
FGRID        = np.linspace(0.0, 1.0, N_STEPS + 1)  # common fraction grid for averaging curves

def segment_image(base_img):
    """base_img: (224,224,3) float [0,1] (denorm output) -> integer superpixel labels (224,224).
    slic's enforce_connectivity merges small superpixels, so the returned count can land well
    under the request on smooth patches — escalate the request until the floor is met."""
    for req in (N_SEGMENTS, N_SEGMENTS * 2, N_SEGMENTS * 4, N_SEGMENTS * 8):
        segs = slic(base_img, n_segments=req, compactness=10, start_label=0, channel_axis=-1)
        if len(np.unique(segs)) >= MIN_SEGMENTS:
            break
    n = len(np.unique(segs))
    assert segs.min() == 0 and n >= MIN_SEGMENTS, f"degenerate segmentation: {n} segments"
    return segs

def seg_order(attr_map, segs):
    """Rank segment ids by mean attribution inside each superpixel, most important first."""
    assert np.isfinite(attr_map).all(), "attribution map has non-finite values"   # 14 Jul rule
    ids = np.unique(segs)
    means = np.array([attr_map[segs == s].mean() for s in ids])
    return ids[np.argsort(-means)]

def irof_curve(sec_patch, branch, target_class, segs, order):
    """Cumulative-deletion probability curve for ONE branch (the other modality held pristine).
    Returns (fractions_removed, p_target); p_target[0] = the unperturbed probability."""
    _, _, d1, d2 = load_pair(sec_patch)
    inp, other = (d1, d2) if branch == 1 else (d2, d1)
    fill = inp.mean(dim=(2, 3), keepdim=True)      # global mean = Stage-4b "removed" convention
    versions, x = [inp], inp
    chunks = np.array_split(order, N_STEPS)
    for chunk in chunks:
        m = torch.from_numpy(np.isin(segs, chunk)).to(inp.device)
        x = x.clone(); x[:, :, m] = fill[:, :, 0]  # (1,3,K) <- (1,3,1) broadcast
        versions.append(x)
    X = torch.cat(versions)                        # (N_STEPS+1, 3, 224, 224)
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), SUB_BATCH):
            xb = X[i:i + SUB_BATCH]
            ob = other.expand(len(xb), -1, -1, -1)
            logits = model(xb, ob) if branch == 1 else model(ob, xb)
            probs.append(F.softmax(logits, 1)[:, target_class].cpu())
    p = torch.cat(probs).numpy()
    assert np.isfinite(p).all(), "non-finite probability in the deletion curve"
    fracs = np.concatenate([[0.0], np.cumsum([len(c) for c in chunks]) / len(order)])
    return fracs, p

def aoc(fracs, p):
    """Area OVER the p/p0 curve — higher = damage front-loaded = more faithful map."""
    return float(1.0 - np.trapz(p / p[0], fracs))

# --- parity anchors on one exemplar (branch 2 = XPL, the dominant modality) ---------------------
sp0, cls0 = exemplars.iloc[0]["sec_patch"], int(exemplars.iloc[0]["model_class"])
_, _, _d1, _d2 = load_pair(sp0)
segs0 = segment_image(denorm(_d2))
_fr, _pc = irof_curve(sp0, 2, cls0, segs0,
                      seg_order(np.random.default_rng(SEED).random((224, 224)), segs0))
_pf, _dp, _dx = modality_ablation(sp0, cls0)
# tolerance 5e-4: batch-8 vs batch-1 CUDA inference differs by fp noise (~1e-5); a wrong fill
# convention would miss by ~1e-2 or more — the assert still catches what it is for
assert abs(_pc[0] - _pf) < 5e-4, f"step 0 != unperturbed prob: {_pc[0]:.6f} vs {_pf:.6f}"
assert abs(_pc[-1] - (_pf - _dx)) < 5e-4, \
    f"full deletion != Stage-4b mean-fill ablation: {_pc[-1]:.6f} vs {_pf - _dx:.6f}"
print(f"anchors OK — p0 {_pc[0]:.3f} == p_full · full deletion {_pc[-1]:.3f} == no-XPL "
      f"{_pf - _dx:.3f} · {len(np.unique(segs0))} superpixels · {len(_fr)} curve points")

In [ ]:
# ── Stage 4c · Step 2: the IROF grid — 5 methods + random, both branches, stratified sample ─────
# [prov: ours | cell-28 pattern | reuses the five validated method functions; adds the random null
#  + AOC table; module swap wraps ONLY the within-branch Chefer, then restores (4b discipline)]
# Fresh seeded sample of correctly-predicted grains from the six key classes. Pass 1 = the four
# no-surgery methods on the pristine model; pass 2 = swap -> within-branch Chefer -> restore.
# Random deletion order (R_REPEATS shuffles, averaged) = the null every method must beat.
# ~12-18 min at N_PER_CLASS_IROF=5 on the 2080 Ti (reasoned, not measured); 2 -> quick ~6 min pass.
import datetime as _dt, json

N_PER_CLASS_IROF = 5
R_REPEATS = 3
rng = np.random.default_rng(SEED + 46)         # stage-4c stream, decoupled from earlier draws

picks = []
for _cls, _nm in KEY_CLASSES.items():
    pool = correct[correct["model_class"] == _cls]
    take = pool.iloc[rng.choice(len(pool), size=min(N_PER_CLASS_IROF, len(pool)), replace=False)]
    picks += [{"sec_patch": r["sec_patch"], "model_class": int(_cls), "name": _nm}
              for _, r in take.iterrows()]
ev = pd.DataFrame(picks)
print(f"IROF sample: {len(ev)} correctly-predicted grains · {ev['name'].nunique()} classes")

METHODS = ["Grad-CAM", "cross-attn", "rollout", "Chefer cross", "Chefer branch"]

# --- pass 1: pristine model — segmentations + the four no-surgery methods' maps -----------------
maps, segs_of = {}, {}
for _, ex in ev.iterrows():
    sp, cls = ex["sec_patch"], int(ex["model_class"])
    _, _, d1, d2 = load_pair(sp)
    segs_of[sp] = (segment_image(denorm(d1)), segment_image(denorm(d2)))
    _, g_p = gradcam_branch(sp, 1, cls); _, g_x = gradcam_branch(sp, 2, cls)
    f_p, f_x = fusion_maps(sp)
    r_p = branch_rollout(sp, "v1", model.vit1); r_x = branch_rollout(sp, "v2", model.vit2)
    c_p, c_x = chefer_cross(sp, cls)
    maps[sp] = {"Grad-CAM": (g_p, g_x), "cross-attn": (f_p, f_x),
                "rollout": (r_p, r_x), "Chefer cross": (c_p, c_x)}

# --- pass 2: module swap ONLY for within-branch Chefer, then restore ----------------------------
swap_in(model.vit1); swap_in(model.vit2)
for _, ex in ev.iterrows():
    sp, cls = ex["sec_patch"], int(ex["model_class"])
    maps[sp]["Chefer branch"] = (chefer_branch(sp, model.vit1, cls),
                                 chefer_branch(sp, model.vit2, cls))
swap_out(model.vit1); swap_out(model.vit2)
print("maps done · model restored to pristine forward")

# --- the curves: 5 methods + random, both branches ----------------------------------------------
rows, curves = [], {}
for gi, (_, ex) in enumerate(ev.iterrows()):
    sp, cls = ex["sec_patch"], int(ex["model_class"])
    for branch in (1, 2):
        br = "PPL" if branch == 1 else "XPL"
        segs = segs_of[sp][branch - 1]
        for meth in METHODS:
            fr, p = irof_curve(sp, branch, cls, segs, seg_order(maps[sp][meth][branch - 1], segs))
            rows.append({"sec_patch": sp, "name": ex["name"], "branch": br,
                         "method": meth, "aoc": aoc(fr, p)})
            curves.setdefault((meth, br), []).append(np.interp(FGRID, fr, p / p[0]))
        r_aocs = []
        for _rep in range(R_REPEATS):
            fr, p = irof_curve(sp, branch, cls, segs, rng.permutation(np.unique(segs)))
            r_aocs.append(aoc(fr, p))
            curves.setdefault(("random", br), []).append(np.interp(FGRID, fr, p / p[0]))
        rows.append({"sec_patch": sp, "name": ex["name"], "branch": br,
                     "method": "random", "aoc": float(np.mean(r_aocs))})
    print(f"  ...{gi + 1}/{len(ev)} grains", flush=True)

irof = pd.DataFrame(rows)
assert np.isfinite(irof["aoc"]).all(), "non-finite AOC"
summary = irof.pivot_table(index="method", columns="branch", values="aoc", aggfunc="mean")
summary["mean"] = summary.mean(1)
summary = summary.sort_values("mean", ascending=False)
rand_row = summary.loc["random"]
print("\nIROF AOC (higher = more faithful map):")
print(summary.round(3).to_string())
print("\nvs the random null (mean-of-branches delta):")
for meth in METHODS:
    d = summary.loc[meth, "mean"] - rand_row["mean"]
    print(f"  {meth:14s} {d:+.3f}  {'beats random' if d > 0 else '⚠ DOES NOT beat random'}")

irof.to_csv(PIPELINE_DIR / "cache" / "irof_aoc_per_grain.csv", index=False)
rec = {"date": _dt.date.today().isoformat(),
       "what": "IROF fidelity scores for the 5-method ladder (per-branch superpixel deletion, "
               "global-mean fill parity-anchored to the Stage-4b ablation; random null baseline).",
       "config": {"n_segments": N_SEGMENTS, "n_steps": N_STEPS, "n_per_class": N_PER_CLASS_IROF,
                  "r_repeats": R_REPEATS, "seed": SEED, "fill": "global per-channel image mean",
                  "checkpoint": "polarvit_multiclass25_a.pth"},
       "n_grains": int(len(ev)), "classes": sorted(ev["name"].unique().tolist()),
       "aoc_mean": {m: {b: round(float(summary.loc[m, b]), 4) for b in ("PPL", "XPL")}
                    for m in summary.index},
       "fractions": [round(float(f), 3) for f in FGRID],
       "mean_curves": {f"{m}|{b}": [round(float(v), 4) for v in np.stack(c).mean(0)]
                       for (m, b), c in curves.items()}}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_irof-fidelity.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"\nsaved -> cache/irof_aoc_per_grain.csv · logged -> runs/{out.name}")

In [ ]:
# ── Stage 4c · Step 3: the quantified ladder — mean deletion curves + AOC summary ───────────────
# [prov: ours | cell-27/28 figure conventions | mean curves per branch + AOC bars; random = null]
PALETTE = {"Grad-CAM": "tab:orange", "cross-attn": "tab:green", "rollout": "tab:purple",
           "Chefer cross": "tab:blue", "Chefer branch": "tab:red", "random": "0.5"}

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for bi, br in enumerate(("PPL", "XPL")):
    for meth in METHODS + ["random"]:
        mu = np.stack(curves[(meth, br)]).mean(0)
        ax[bi].plot(FGRID, mu, color=PALETTE[meth], lw=2,
                    ls="--" if meth == "random" else "-",
                    label=f"{meth} (AOC {summary.loc[meth, br]:.3f})")
    ax[bi].set_title(f"{br} deleted (other modality pristine)")
    ax[bi].set_xlabel("fraction of superpixels deleted (most important first)")
    ax[bi].set_ylabel("p(true class) / p0")
    ax[bi].legend(fontsize=8); ax[bi].set_ylim(bottom=0)

y = np.arange(len(summary.index)); h = 0.4
ax[2].barh(y - h / 2, summary["PPL"], h, label="PPL", color="tab:orange")
ax[2].barh(y + h / 2, summary["XPL"], h, label="XPL", color="tab:blue")
ax[2].set_yticks(y); ax[2].set_yticklabels(summary.index, fontsize=9)
ax[2].axvline(summary.loc["random", "mean"], color="k", lw=0.8, ls="--", label="random (mean)")
ax[2].set_xlabel("IROF AOC (higher = more faithful)")
ax[2].set_title("The ladder, quantified")
ax[2].legend(fontsize=8, loc="lower right")
ax[2].invert_yaxis()
fig.suptitle("IROF: deleting what each method ranks important — faithful maps collapse the "
             "prediction fastest", fontsize=12)
plt.tight_layout(); plt.show()

xpl_lead = summary.loc[METHODS, "XPL"].mean() - summary.loc[METHODS, "PPL"].mean()
verdict = ("XPL deletions hurt more — echoes the Stage-4b colour-dominance read" if xpl_lead > 0
           else "PPL-side deletions hurt more — DISCORDANT with Stage 4b, flag it")
print(f"mean method AOC, XPL − PPL = {xpl_lead:+.3f}  ({verdict})")

## Stage 4d — sanity check: do the maps depend on the model?

Adebayo et al. 2018: randomise weights output-backwards and demand each attribution map degrade.
Two rungs (decoder+fusion+head → everything), similarity = abs-map Spearman + SSIM vs the
pristine cell-31 maps. Rollout never reads the decoder/head, so its rung-1 invariance is the
designed control — and explains its IROF ≤-random verdict mechanistically

In [ ]:
# ── Stage 4d · Step 1: sanity-check machinery — parameter randomisation (Adebayo et al. 2018) ───
# [prov: lit | Adebayo et al. 2018, arXiv:1810.03292 (sanity checks via parameter randomisation) |
#  impl ours: two-rung cascade fitted to the two-stream model (head+decoder -> everything),
#  name-filtered so no architecture attribute names are assumed; similarity = abs-map Spearman
#  (rank order is what deletion metrics consume) + SSIM; pristine restored + parity-asserted]
import copy
from scipy.stats import spearmanr
from skimage.metrics import structural_similarity as _ssim

PRISTINE_SD = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
ALL_PARAMS  = [n for n, _ in model.named_parameters()]
VIT_PARAMS  = [n for n in ALL_PARAMS if n.startswith(("vit1.", "vit2."))]
HEAD_PARAMS = [n for n in ALL_PARAMS if n not in VIT_PARAMS]  # decoder + fusion + head = read-out
assert VIT_PARAMS and HEAD_PARAMS, "name filter found nothing — inspect model.named_parameters()"

RUNGS = [("pristine", []), ("head+decoder", HEAD_PARAMS), ("all", ALL_PARAMS)]  # cascade

def load_randomised(names, seed):
    """Restore pristine weights, then re-draw `names` from a seeded N(0, std_pristine) per
    tensor. Scale-preserving re-init keeps activations in range so maps degrade for model
    reasons, not float overflow. Empty `names` -> pristine restore only."""
    model.load_state_dict(PRISTINE_SD)
    if not names:
        return
    g = torch.Generator(device="cpu").manual_seed(seed)
    sd = model.state_dict()
    for n in names:
        t = PRISTINE_SD[n]
        std = float(t.std()) if t.numel() > 1 and float(t.std()) > 0 else 0.02
        sd[n] = torch.empty_like(t, device="cpu").normal_(0.0, std, generator=g)
    model.load_state_dict(sd)

def map_sim(a, b):
    """Similarity of two attribution maps: Spearman rank corr of |values| + SSIM on
    min-max-normalised maps. A constant (degenerate) map scores 0, not NaN."""
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    rho = spearmanr(np.abs(a).ravel(), np.abs(b).ravel()).statistic
    if not np.isfinite(rho):
        rho = 0.0
    na = (a - a.min()) / (a.max() - a.min() + 1e-12)
    nb = (b - b.min()) / (b.max() - b.min() + 1e-12)
    return float(rho), float(_ssim(na, nb, data_range=1.0))

def maps_for(sp, cls):
    """All five ladder maps for one grain on the CURRENT weights (cell-31 pattern, one grain).
    Module swap wraps ONLY the within-branch Chefer, then restores (4b discipline)."""
    _, g_p = gradcam_branch(sp, 1, cls); _, g_x = gradcam_branch(sp, 2, cls)
    f_p, f_x = fusion_maps(sp)
    r_p = branch_rollout(sp, "v1", model.vit1); r_x = branch_rollout(sp, "v2", model.vit2)
    c_p, c_x = chefer_cross(sp, cls)
    swap_in(model.vit1); swap_in(model.vit2)
    b_p, b_x = chefer_branch(sp, model.vit1, cls), chefer_branch(sp, model.vit2, cls)
    swap_out(model.vit1); swap_out(model.vit2)
    return {"Grad-CAM": (g_p, g_x), "cross-attn": (f_p, f_x), "rollout": (r_p, r_x),
            "Chefer cross": (c_p, c_x), "Chefer branch": (b_p, b_x)}

# --- anchors on one exemplar --------------------------------------------------------------------
sp0, cls0 = ev.iloc[0]["sec_patch"], int(ev.iloc[0]["model_class"])
_, _, _d1, _d2 = load_pair(sp0)
with torch.no_grad():
    _logit_ref = model(_d1, _d2).cpu()
m0 = maps_for(sp0, cls0)                       # pristine recompute must match cell 31's maps
for _meth in METHODS:
    _rho, _s = map_sim(m0[_meth][1], maps[sp0][_meth][1])
    assert _rho > 0.999, f"{_meth}: pristine recompute != cell-31 map (spearman {_rho:.4f})"
load_randomised(HEAD_PARAMS, SEED + 47)        # round-trip: randomise -> restore -> parity
load_randomised([], 0)
with torch.no_grad():
    _logit_back = model(_d1, _d2).cpu()
assert torch.allclose(_logit_ref, _logit_back, atol=1e-5), "restore parity failed"
print(f"anchors OK — 5 pristine recomputes match cell 31 (ρ>0.999) · randomise->restore parity "
      f"{float((_logit_ref - _logit_back).abs().max()):.2e} · rungs: "
      + " · ".join(f"{n} ({len(p)} tensors)" for n, p in RUNGS))

In [ ]:
# ── Stage 4d · Step 2: the cascade — 5 methods × 2 branches × 2 randomised rungs ────────────────
# [prov: ours | reuses cell-31 pristine maps as the reference; cascade seeded SEED+47]
# Expected structure (reasoned): class-specific maps (Grad-CAM, both Chefers) and the
# decoder-read cross-attn degrade at rung 1; rollout is rung-1 INVARIANT by construction (never
# reads decoder/head — the designed control) and degrades only at rung 2.
import datetime as _dt, json

N_SANITY = 3                                   # grains per class from the IROF sample -> 18
sv = ev.groupby("name", group_keys=False).head(N_SANITY)
print(f"sanity sample: {len(sv)} grains (reusing the IROF stratified sample)")

rows = []
for rung, names in RUNGS[1:]:                  # pristine similarity == 1 by the Step-1 anchor
    load_randomised(names, SEED + 47)
    for gi, (_, ex) in enumerate(sv.iterrows()):
        sp, cls = ex["sec_patch"], int(ex["model_class"])
        mr = maps_for(sp, cls)
        for meth in METHODS:
            for bi, br in ((0, "PPL"), (1, "XPL")):
                rho, s = map_sim(mr[meth][bi], maps[sp][meth][bi])
                rows.append({"rung": rung, "method": meth, "branch": br, "sec_patch": sp,
                             "name": ex["name"], "spearman": rho, "ssim": s})
    print(f"  rung '{rung}' done", flush=True)
load_randomised([], 0)                          # pristine restored for everything downstream
with torch.no_grad():
    assert torch.allclose(model(_d1, _d2).cpu(), _logit_ref, atol=1e-5), "model NOT restored"
print("model restored to pristine weights (parity re-asserted)")

san = pd.DataFrame(rows)
assert len(san) == 2 * len(sv) * len(METHODS) * 2, "row count off"
assert np.isfinite(san[["spearman", "ssim"]]).all().all(), "non-finite similarity"
piv = san.pivot_table(index="method", columns="rung", values="spearman", aggfunc="mean")
piv = piv[["head+decoder", "all"]].reindex(METHODS)
print("\nabs-map Spearman vs pristine (1.0 = unchanged; LOW = the map depends on the weights):")
print(piv.round(3).to_string())
for meth in ("Grad-CAM", "Chefer branch", "Chefer cross", "cross-attn"):
    ok = piv.loc[meth, "head+decoder"] < 0.9
    print(f"  {meth:14s} rung-1 "
          + ("degrades — PASSES the sanity check" if ok
             else "⚠ INVARIANT under head+decoder randomisation — investigate"))
print(f"  rollout        rung-1 ρ {piv.loc['rollout', 'head+decoder']:.3f} — expected ≈1 "
      f"(never reads the decoder/head; the designed control + the IROF ≤-random echo)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for bi, br in enumerate(("PPL", "XPL")):
    sub = san[san["branch"] == br].pivot_table(index="method", columns="rung",
                                               values="spearman", aggfunc="mean")
    for meth in METHODS:
        ax[bi].plot([0, 1, 2], [1.0, sub.loc[meth, "head+decoder"], sub.loc[meth, "all"]],
                    marker="o", label=meth)
    ax[bi].set_xticks([0, 1, 2])
    ax[bi].set_xticklabels(["pristine", "head+decoder\nrandomised", "all\nrandomised"])
    ax[bi].set_ylabel("abs-map Spearman vs pristine")
    ax[bi].set_title(f"{br} branch"); ax[bi].axhline(0, color="k", lw=0.5)
    ax[bi].legend(fontsize=8)
fig.suptitle("Sanity check (Adebayo et al. 2018): explanations must depend on the model")
plt.tight_layout(); plt.show()

san.to_csv(PIPELINE_DIR / "cache" / "sanity_randomisation_per_grain.csv", index=False)
rec = {"date": _dt.date.today().isoformat(),
       "what": "Parameter-randomisation sanity check (Adebayo 2018) over the 5-method ladder: "
               "two-rung cascade (head+decoder -> all), abs-map Spearman + SSIM vs pristine.",
       "config": {"n_grains": int(len(sv)), "rungs": [r for r, _ in RUNGS[1:]],
                  "seed": SEED + 47, "reference": "cell-31 pristine maps",
                  "checkpoint": "polarvit_multiclass25_a.pth"},
       "spearman_mean": {m: {r: round(float(piv.loc[m, r]), 4) for r in piv.columns}
                         for m in piv.index},
       "ssim_mean": {m: round(float(san[san["method"] == m]["ssim"].mean()), 4)
                     for m in METHODS}}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_sanity-randomisation.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"\nsaved -> cache/sanity_randomisation_per_grain.csv · logged -> runs/{out.name}")

## Stage 4e — SpRAy: which relevance strategies exist across the sample?

Lapuschkin et al. 2019, sample-scale: cluster L1-normalised 28×28 signatures of the Chefer-branch
maps (the top-fidelity rung) with seeded k-means, k by silhouette. The read: clusters by CLASS =
per-mineral strategies; clusters by LOTE = an acquisition strategy — the explanation-level echo
of the 0.973 latent lote-AUC.

In [ ]:
# ── Stage 4e · Step 1: SpRAy-style strategy clustering of Chefer-branch maps (sample-scale) ─────
# [prov: lit | Lapuschkin et al. 2019, Nat. Commun. (SpRAy) | impl ours: sample-scale per the
#  17 Jul scope call; Chefer branch = the top-fidelity rung (IROF 14 Jul); 28x28 block-mean
#  L1 signatures; seeded k-means, k chosen by silhouette over a small grid]
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_PER_CLASS_SPRAY = 10
K_GRID = (2, 3, 4, 5, 6)
rng_sp = np.random.default_rng(SEED + 48)      # stage-4e stream, decoupled from earlier draws

picks = []
for _cls, _nm in KEY_CLASSES.items():
    pool = correct[correct["model_class"] == _cls]
    take = pool.iloc[rng_sp.choice(len(pool), size=min(N_PER_CLASS_SPRAY, len(pool)),
                                   replace=False)]
    picks += [{"sec_patch": r["sec_patch"], "model_class": int(_cls), "name": _nm}
              for _, r in take.iterrows()]
spr = pd.DataFrame(picks).reset_index(drop=True)
spr["lote"] = ["lote2" if "LOTE2" in sp else "lote1" for sp in spr["sec_patch"]]
assert spr["lote"].nunique() >= 1, "lote parse failed — check sec_patch path format"
print(f"SpRAy sample: {len(spr)} grains · {spr['name'].nunique()} classes · lote mix:",
      {k: int(v) for k, v in spr["lote"].value_counts().items()})

def spray_vec(m):
    """224x224 map -> 28x28 block mean -> abs -> L1-normalised flat vector: a shape signature
    (WHERE relevance sits), magnitude-free so bright maps don't dominate the geometry."""
    v = np.abs(np.asarray(m, dtype=np.float64)).reshape(28, 8, 28, 8).mean(axis=(1, 3)).ravel()
    s = v.sum()
    assert np.isfinite(s) and s > 0, "degenerate map"
    return v / s

swap_in(model.vit1); swap_in(model.vit2)       # one swap around the whole collection pass
smaps = {}
for gi, (_, ex) in enumerate(spr.iterrows()):
    sp, cls = ex["sec_patch"], int(ex["model_class"])
    smaps[sp] = (chefer_branch(sp, model.vit1, cls), chefer_branch(sp, model.vit2, cls))
    if (gi + 1) % 10 == 0:
        print(f"  ...{gi + 1}/{len(spr)} grains", flush=True)
swap_out(model.vit1); swap_out(model.vit2)
print("maps done · model restored to pristine forward")

results = {}
for bi, br in ((0, "PPL"), (1, "XPL")):
    X = np.stack([spray_vec(smaps[sp][bi]) for sp in spr["sec_patch"]])
    sil = {}
    for k in K_GRID:
        if k >= len(X):
            break
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
        sil[k] = float(silhouette_score(X, km.labels_))
    k_best = max(sil, key=sil.get)
    km = KMeans(n_clusters=k_best, n_init=10, random_state=SEED).fit(X)
    results[br] = {"X": X, "labels": km.labels_, "centers": km.cluster_centers_,
                   "k": k_best, "sil": sil}
    lab = pd.Series(km.labels_, index=spr.index, name="cluster")
    print(f"\n{br}: k={k_best} (silhouette {sil[k_best]:.3f}; grid "
          + ", ".join(f"k{k}={v:.3f}" for k, v in sil.items()) + ")")
    print(pd.crosstab(lab, spr["name"]).to_string())
    print(pd.crosstab(lab, spr["lote"]).to_string())

In [ ]:
# ── Stage 4e · Step 2: cluster exemplars — what does each strategy look at? ─────────────────────
# [prov: ours | cell-28 overlay conventions | nearest-centroid exemplars per cluster, XPL shown
#  (the dominant modality); montage saved to code/figures for Thu 23]
import datetime as _dt, json

FIG_DIR = PIPELINE_DIR.parent / "figures"
assert FIG_DIR.is_dir(), f"figures dir missing: {FIG_DIR}"
BR_SHOW, BI_SHOW = "XPL", 1
res, n_ex = results[BR_SHOW], 4

fig, axes = plt.subplots(res["k"], n_ex, figsize=(3 * n_ex, 3 * res["k"]), squeeze=False)
comp = {}
for c in range(res["k"]):
    idx = np.where(res["labels"] == c)[0]
    d = np.linalg.norm(res["X"][idx] - res["centers"][c], axis=1)
    show = idx[np.argsort(d)[:n_ex]]
    comp[c] = {"n": int(len(idx)),
               "classes": {k: int(v) for k, v in spr.iloc[idx]["name"].value_counts().items()},
               "lotes": {k: int(v) for k, v in spr.iloc[idx]["lote"].value_counts().items()}}
    for j in range(n_ex):
        axx = axes[c][j]
        if j < len(show):
            ex = spr.iloc[show[j]]
            _, _, d1, d2 = load_pair(ex["sec_patch"])
            img = denorm(d2 if BI_SHOW == 1 else d1)
            m = np.asarray(smaps[ex["sec_patch"]][BI_SHOW], dtype=np.float64)
            m = (m - m.min()) / (m.max() - m.min() + 1e-12)
            axx.imshow(img); axx.imshow(m, cmap="jet", alpha=0.35)
            axx.set_title(f"c{c} · {ex['name']} · {ex['lote']}", fontsize=8)
        axx.axis("off")
fig.suptitle(f"SpRAy clusters ({BR_SHOW} Chefer-branch maps): relevance strategies, "
             f"nearest-centroid exemplars")
plt.tight_layout()
figpath = FIG_DIR / f"{_dt.date.today().isoformat()}_spray-clusters-{BR_SHOW.lower()}.png"
fig.savefig(figpath, dpi=150, bbox_inches="tight"); plt.show()

rec = {"date": _dt.date.today().isoformat(),
       "what": "SpRAy-style clustering of Chefer-branch relevance maps (sample-scale): 28x28 L1 "
               "signatures, seeded k-means, silhouette-chosen k; composition by class + lote.",
       "config": {"n_per_class": N_PER_CLASS_SPRAY, "n_grains": int(len(spr)),
                  "k_grid": list(K_GRID), "seed": SEED + 48, "method": "Chefer branch",
                  "downsample": "28x28 block mean + L1", "checkpoint": "polarvit_multiclass25_a.pth"},
       "branches": {br: {"k": int(r["k"]),
                         "silhouette": {str(k): round(v, 4) for k, v in r["sil"].items()},
                         "cluster_sizes": [int((r["labels"] == c).sum()) for c in range(r["k"])]}
                    for br, r in results.items()},
       "composition_shown_branch": {str(c): comp[c] for c in comp},
       "figure": figpath.name}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_spray-clusters.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"saved figure -> figures/{figpath.name} · logged -> runs/{out.name}")

In [ ]:
# ── Stage 5 · Step 1: Stage-6 paired sample → per-branch + fused CLS embeddings (frozen, cached) ──
# Rebuilds the EXACT Stage-6 sample (CAP=80/class, random_state=SEED) so the after-5.3.2 re-run
# compares like-for-like, then captures three 1024-D CLS vectors per grain via read-only forward
# hooks on the two decoders: PPL branch (vit1, post-cross-attn), XPL branch (vit2), and the fused
# α-combined feature the classifier actually reads. Deterministic; ~2–4 min of frozen passes.
import numpy as np, pandas as pd, torch

EMB_CACHE = PIPELINE_DIR / "cache" / "latent_embeddings_stage6.npz"
CAP, BATCH = 80, 32

# --- the paired sample: identical grains to pipeline.ipynb Stage 6 Step 2 --------------------------
samp = (f2.groupby("model_class", group_keys=False)[f2.columns]
          .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED)))
samp = samp.reset_index(drop=True)
samp["lote"]  = samp["sec_patch"].str.split("/").str[0]
samp["slide"] = samp["sec_patch"].str.split("/").str[:2].str.join("/")
print(f"paired sample: {len(samp):,} grains · {samp['model_class'].nunique()} classes · "
      f"{samp['slide'].nunique()} slides · lotes {samp['lote'].value_counts().to_dict()}")

if EMB_CACHE.exists():
    d = np.load(EMB_CACHE, allow_pickle=True)
    emb = {k: d[k] for k in ("ppl", "xpl", "fused")}
    meta = pd.DataFrame({k: d[k] for k in ("sec_patch", "model_class", "name", "lote", "slide")})
    print(f"loaded cached embeddings: {EMB_CACHE.name}  ({len(meta):,} grains)")
else:
    # --- read-only forward hooks on the two decoders (capture output[0] = features, [:,0] = CLS) --
    grab = {}
    def hook(tag):
        def _h(_m, _in, out):
            feats = out[0] if isinstance(out, tuple) else out      # Decoder returns (feats, attns)
            grab[tag] = feats[:, 0].detach().float().cpu()         # CLS token, (B, 1024)
        return _h
    h1 = model.decoder_1.register_forward_hook(hook("ppl"))        # vit1/PPL branch post-cross-attn
    h2 = model.decoder_2.register_forward_hook(hook("xpl"))        # vit2/XPL branch post-cross-attn
    alpha = torch.sigmoid(model.alpha.detach()).item()
    print(f"fusion α = {alpha:.3f}  (fused = α·PPL + (1−α)·XPL, matching forward)")

    def load_d(sec_patch):
        ppl, xpl = load_patch_images(sec_patch)
        return preprocess(to_tensor(ppl)), preprocess(to_tensor(xpl))

    ppl_e, xpl_e, fused_e = [], [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(samp), BATCH):
            chunk = samp["sec_patch"].iloc[i:i + BATCH].tolist()
            pairs = [load_d(sp) for sp in chunk]
            d1 = torch.stack([p for p, _ in pairs]).to(device)
            d2 = torch.stack([x for _, x in pairs]).to(device)
            _ = model(d1, d2)                                      # hooks fire; logits unused
            p, x = grab["ppl"], grab["xpl"]
            ppl_e.append(p); xpl_e.append(x)
            fused_e.append(alpha * p + (1 - alpha) * x)
            if i % (BATCH * 10) == 0:
                print(f"  ...{i}/{len(samp)}", flush=True)
    h1.remove(); h2.remove()                                       # ALWAYS un-hook (leave model pristine)

    emb = {"ppl": torch.cat(ppl_e).numpy(), "xpl": torch.cat(xpl_e).numpy(),
           "fused": torch.cat(fused_e).numpy()}
    meta = samp[["sec_patch", "model_class", "name", "lote", "slide"]] if "name" in samp \
           else samp.assign(name=samp["multiclass_name"])[["sec_patch", "model_class", "name", "lote", "slide"]]
    np.savez_compressed(EMB_CACHE, **emb, **{c: meta[c].to_numpy() for c in meta.columns})
    print(f"cached → {EMB_CACHE.name}  (fused/ppl/xpl each {emb['fused'].shape})")

# sanity: fused must sit on the α-line between the two branch features (not a coding slip)
_a = torch.sigmoid(model.alpha.detach()).item()
_chk = np.abs(emb["fused"] - (_a * emb["ppl"] + (1 - _a) * emb["xpl"])).max()
print(f"fusion identity check: max|fused − (α·ppl+(1−α)·xpl)| = {_chk:.2e}  (want ~0)")

In [ ]:
# ── Stage 5 · Step 2: reduce + plot (fused/PPL/XPL × by-class/by-lote) + quantitative lote probe ──
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

try:
    import umap
    def reduce2d(X):
        p = PCA(n_components=50, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))
        return umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=SEED).fit_transform(p), "UMAP"
except ImportError:
    from sklearn.manifold import TSNE
    def reduce2d(X):
        p = PCA(n_components=50, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))
        return TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(p), "t-SNE"

y_cls  = meta["model_class"].to_numpy()
y_lote = (meta["lote"].to_numpy() == meta["lote"].unique()[0]).astype(int)   # binary lote label
base_rate = max(y_lote.mean(), 1 - y_lote.mean())

SPACES = [("fused", "Fused (post-α) — what the classifier reads"),
          ("ppl",   "PPL branch CLS (vit1)"),
          ("xpl",   "XPL branch CLS (vit2)")]

# --- the lote linear-probe: does acquisition batch leak into each embedding? (the calibration number)
print(f"lote linear-probe — 5-fold CV accuracy vs base rate {base_rate:.3f} "
      f"(>> base rate ⇒ acquisition batch leaks into the embedding):")
probe = {}
for key, _ in SPACES:
    Xs = StandardScaler().fit_transform(emb[key])
    acc = cross_val_score(LogisticRegression(max_iter=2000, C=1.0),
                          Xs, y_lote, cv=5, scoring="accuracy").mean()
    probe[key] = acc
    print(f"  {key:6s}: {acc:.3f}   (+{acc - base_rate:+.3f} over base)")

# --- the 3×2 figure (rows = embedding spaces, cols = colour by class / by lote) -------------------
fig, axes = plt.subplots(len(SPACES), 2, figsize=(13, 4.4 * len(SPACES)))
for r, (key, title) in enumerate(SPACES):
    XY, algo = reduce2d(emb[key])
    axes[r, 0].scatter(XY[:, 0], XY[:, 1], c=y_cls, cmap="tab20", s=6, alpha=0.7)
    axes[r, 0].set_title(f"{title}\n{algo}, coloured by mineral class (25)")
    for lv, lab, col in [(0, meta["lote"].unique()[0], "#1f77b4"),
                         (1, meta["lote"].unique()[-1], "#d62728")]:
        m = y_lote == (0 if lab == meta["lote"].unique()[0] else 1)
        axes[r, 1].scatter(XY[m, 0], XY[m, 1], c=col, s=6, alpha=0.6, label=str(lab))
    axes[r, 1].set_title(f"{title}\ncoloured by lote — probe acc {probe[key]:.2f} vs base {base_rate:.2f}")
    axes[r, 1].legend(markerscale=2, fontsize=8)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
fig.suptitle("Frozen baseline latent geometry (BEFORE constraints) — re-run after 5.3.2 for the "
             "before/after figure", y=1.005, fontsize=12)
plt.tight_layout(); plt.show()

# --- log the probe numbers (the calibration argument, as data) -----------------------------------
import json, datetime
rec = {"date": datetime.date.today().isoformat(),
       "what": "Frozen-baseline latent-space lote linear-probe (embedding-level calibration argument), "
               "before any constraint — paired Stage-6 sample.",
       "n_grains": int(len(meta)), "n_slides": int(meta["slide"].nunique()),
       "lote_base_rate": round(float(base_rate), 3),
       "lote_probe_acc": {k: round(float(v), 3) for k, v in probe.items()},
       "fusion_alpha": round(float(torch.sigmoid(model.alpha.detach()).item()), 3)}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_latent-lote-probe.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"\nlogged → {out.name}")

In [ ]:
# ── Stage 5 · Step 3: re-score the lote probe with AUC + balanced accuracy (imbalance-robust) ─────
# Plain accuracy is metric-limited at the 88/12 lote split (base 0.879). ROC-AUC and balanced
# accuracy are prior-invariant, so they read the batch separability the accuracy ceiling masks.
# Reads the cached embeddings only — no model, no GPU.
import numpy as np, json, datetime
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

d = np.load(EMB_CACHE, allow_pickle=True) if "emb" not in dir() else None
E = emb if d is None else {k: d[k] for k in ("ppl", "xpl", "fused")}
LOTE = (meta["lote"].to_numpy() == meta["lote"].unique()[0]).astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
print(f"lote {dict(meta['lote'].value_counts())} · base {max(LOTE.mean(), 1-LOTE.mean()):.3f}\n"
      f"{'space':6s} {'AUC':>6s} {'bal-acc':>8s}   (AUC 0.5 = no batch leakage · 1.0 = fully encoded)")
scores = {}
for k in ("fused", "ppl", "xpl"):
    Xs = StandardScaler().fit_transform(E[k])
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")   # balanced: don't fold to majority
    auc = cross_val_score(lr, Xs, LOTE, cv=cv, scoring="roc_auc").mean()
    bac = cross_val_score(lr, Xs, LOTE, cv=cv, scoring="balanced_accuracy").mean()
    scores[k] = {"auc": round(float(auc), 3), "bal_acc": round(float(bac), 3)}
    print(f"{k:6s} {auc:6.3f} {bac:8.3f}")

verdict = ("STRONG batch leakage — the embedding encodes acquisition, not just mineralogy → the "
           "embedding-level calibration argument holds" if scores["fused"]["auc"] >= 0.75 else
           "MODERATE leakage" if scores["fused"]["auc"] >= 0.65 else
           "WEAK/none — batch structure barely present in the latent space (lean the case on the "
           "input-level δ̂ evidence instead)")
print(f"\nfused AUC {scores['fused']['auc']:.3f} → {verdict}")
print(f"per-branch: PPL {scores['ppl']['auc']:.3f} vs XPL {scores['xpl']['auc']:.3f} — "
      f"{'both carry it (pervasive across streams, echoes the 6-Jul shift finding)' if min(scores['ppl']['auc'], scores['xpl']['auc']) >= 0.65 else 'asymmetric — the higher branch dominates batch encoding'}")

# ── log it: a NEW dated record (see the note below — do NOT mutate the 7 Jul record) ──────────────
rec = {"date": datetime.date.today().isoformat(),
       "what": "Latent lote linear-probe re-scored with imbalance-robust metrics (ROC-AUC + balanced "
               "accuracy). Plain accuracy is metric-limited at the 88/12 lote split (base 0.879); "
               "AUC/bal-acc are the read of record for batch separability.",
       "n_grains": int(len(meta)), "lote_base_rate": round(float(max(LOTE.mean(), 1 - LOTE.mean())), 3),
       "lote_probe_auc": scores,
       "source_embeddings": EMB_CACHE.name,
       "note": "companion to lote_probe_acc in the 7 Jul record; supersedes it as the read of record."}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_latent-lote-probe-auc.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"\nlogged → {out.name}")

In [ ]:
# ── Stage 5 · Step 4: within-class lote probe — control for the lote↔mineralogy confound (CPU) ────
# Fix the mineral class → any residual lote separability = ACQUISITION, not composition. Runs on
# every class with ≥12 grains of the minority lote (folds = min(5, n_minority)). Cache-only.
import numpy as np, json, datetime
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

name_of = meta.groupby("model_class")["name"].agg(lambda s: s.mode().iat[0]).to_dict()
cnt = meta.groupby(["model_class", "lote"]).size().unstack(fill_value=0); cnt["min_lote"] = cnt.min(1)
qual = cnt[cnt["min_lote"] >= 12].sort_values("min_lote", ascending=False)
LOTE = (meta["lote"].to_numpy() == meta["lote"].unique()[0]).astype(int)

print(f"WITHIN-CLASS lote probe (mineralogy fixed → residual AUC = acquisition only)")
print(f"{'class':20s} {'n':>4s} {'n_min':>6s} {'fused':>6s} {'ppl':>6s} {'xpl':>6s}")
per, fused_aucs = {}, []
for cls in qual.index:
    m = meta["model_class"].to_numpy() == cls
    y = LOTE[m]; nmin = int(min((y == 0).sum(), (y == 1).sum())); k = min(5, nmin)
    if k < 3: continue
    cv = StratifiedKFold(k, shuffle=True, random_state=SEED)
    a = {key: cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"),
                              StandardScaler().fit_transform(emb[key][m]), y, cv=cv,
                              scoring="roc_auc").mean() for key in ("fused", "ppl", "xpl")}
    per[name_of[cls]] = {"n": int(m.sum()), "n_min": nmin, **{k2: round(float(v), 3) for k2, v in a.items()}}
    fused_aucs.append(a["fused"])
    print(f"{name_of[cls][:20]:20s} {m.sum():4d} {nmin:6d} {a['fused']:6.3f} {a['ppl']:6.3f} {a['xpl']:6.3f}")

mean_fused = float(np.mean(fused_aucs))
print(f"\nwithin-class fused AUC: mean {mean_fused:.3f} · range [{min(fused_aucs):.3f}, {max(fused_aucs):.3f}]"
      f" over {len(fused_aucs)} classes   (cross-class was 0.973)")
print("VERDICT:", "acquisition, NOT composition — the confound is defeated, calibration argument airtight"
      if mean_fused >= 0.75 else "partial — composition inflated the cross-class number"
      if mean_fused >= 0.62 else "collapses within-class — cross-class signal was mostly mineralogy")

# Write a NEW dated record — never mutate an archived one. The 7 Jul file is a dated historical
# artefact that also ships on the hand-over drive, so a read-modify-write would silently diverge
# repo from drive. Convention: f"{date}_..." exactly like every other record writer (fixed 1 Aug).
_today = datetime.date.today().isoformat()
_dst = PIPELINE_DIR / "runs" / f"{_today}_latent-lote-probe.json"
if _dst.exists():                      # Step 2 already wrote today's record -> augment it
    rec = json.loads(_dst.read_text(encoding="utf-8"))
else:                                  # seed (READ-ONLY) from the newest existing record
    _src = sorted((PIPELINE_DIR / "runs").glob("*_latent-lote-probe.json"))
    assert _src, "no latent-lote-probe record to seed from — run Stage 5 Step 2 first"
    rec = json.loads(_src[-1].read_text(encoding="utf-8"))
    rec["seeded_from"], rec["date"] = _src[-1].name, _today
rec["within_class_probe"] = {"classes": per, "mean_fused_auc": round(mean_fused, 3)}
_dst.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"run log written -> {_dst.name} (within_class_probe added; archived records untouched)")

In [ ]:
# ── Stage 5 · Step 5: before/after latent geometry — the constraint's effect on lote structure ──
# [prov: ours | Step 3/4's probe protocol re-applied unchanged (balanced LR, StratifiedKFold,
#  ROC-AUC) | reads the two caches only — no model, no GPU; safe alongside any kernel]
# BEFORE = the 7 Jul frozen-baseline cache (fused AUC 0.973 / within-class 0.900 of record).
# AFTER = a constrained checkpoint's cache (pipeline.ipynb Step 7·16). Same grains (intersection
# — the after is 5.3.1-masked), same capture points, same probe → any AUC movement = the training.
import json, datetime
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

if "PIPELINE_DIR" not in globals():    # fresh-kernel affordance (caught live 16 Jul): this cell
    PIPELINE_DIR = Path.cwd()          # is cache-only, so it must run WITHOUT the XAI setup cells
    assert (PIPELINE_DIR / "cache").exists(), \
        "PIPELINE_DIR not set and cwd is not code/pipeline — run the notebook's setup cells"
if "SEED" not in globals():
    SEED = 0                           # the pipeline seed of record

AFTER_CACHE = None                     # None → newest latent_embeddings_after_*.npz

_before = np.load(PIPELINE_DIR / "cache" / "latent_embeddings_stage6.npz", allow_pickle=True)
_afters = sorted((PIPELINE_DIR / "cache").glob("latent_embeddings_after_*.npz"),
                 key=lambda p: p.stat().st_mtime)
assert _afters or AFTER_CACHE, "no after-cache — run pipeline.ipynb Step 7·16 first"
_after = np.load(Path(AFTER_CACHE) if AFTER_CACHE else _afters[-1], allow_pickle=True)
_acfg = json.loads(str(_after["config"]))
print(f"before: latent_embeddings_stage6.npz ({len(_before['sec_patch']):,} grains) · "
      f"after: {(_afters[-1].name if not AFTER_CACHE else AFTER_CACHE)} "
      f"({len(_after['sec_patch']):,} grains, ckpt {_acfg.get('checkpoint')})")

_b_idx = {p: i for i, p in enumerate(_before["sec_patch"])}
common = [p for p in _after["sec_patch"] if p in _b_idx]     # after ⊆ before (mask) — assert big
assert len(common) == len(_after["sec_patch"]), "after-grains missing from the before capture"
bi = np.array([_b_idx[p] for p in common])
E = {"before": {k: _before[k][bi] for k in ("ppl", "xpl", "fused")},
     "after":  {k: np.asarray(_after[k]) for k in ("ppl", "xpl", "fused")}}
lote = np.asarray(_after["lote"]); y = (lote == np.unique(lote)[0]).astype(int)
cls = np.asarray(_after["model_class"]); names = np.asarray(_after["name"])
base = max(y.mean(), 1 - y.mean())

def lote_auc(X):
    return float(cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"),
                                 StandardScaler().fit_transform(X), y,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                 scoring="roc_auc").mean())

def within_class_mean(Emb):
    aucs = []
    for c in np.unique(cls):
        m = cls == c; yc = y[m]
        nmin = int(min(yc.sum(), (1 - yc).sum()))
        if nmin < 12:
            continue
        cv = StratifiedKFold(min(5, nmin), shuffle=True, random_state=SEED)
        aucs.append(float(cross_val_score(
            LogisticRegression(max_iter=2000, class_weight="balanced"),
            StandardScaler().fit_transform(Emb["fused"][m]), yc,
            cv=cv, scoring="roc_auc").mean()))
    return (float(np.mean(aucs)), len(aucs)) if aucs else (float("nan"), 0)

print(f"\nlote probe on the SAME {len(common):,} masked grains (base rate {base:.3f}):")
tbl = {}
for cond in ("before", "after"):
    tbl[cond] = {k: round(lote_auc(E[cond][k]), 3) for k in ("fused", "ppl", "xpl")}
    wc, nwc = within_class_mean(E[cond])
    tbl[cond]["within_class_fused"] = round(wc, 3)
    print(f"  {cond:6s}: fused {tbl[cond]['fused']:.3f} · ppl {tbl[cond]['ppl']:.3f} · "
          f"xpl {tbl[cond]['xpl']:.3f} · within-class fused {wc:.3f} ({nwc} classes)")
_d = tbl["after"]["fused"] - tbl["before"]["fused"]
verdict = ("constrained training REDUCED acquisition structure in the representation — the "
           "latent-level payoff" if _d <= -0.05 else
           "little movement — the constraint reshapes decisions more than geometry (report as-is)"
           if abs(_d) < 0.05 else
           "acquisition structure INCREASED — investigate before writing up")
print(f"\nfused Δ {_d:+.3f} → {verdict}")

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
def _2d(X):
    p = PCA(n_components=min(50, X.shape[1]), random_state=SEED).fit_transform(
        StandardScaler().fit_transform(X))
    return TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(p)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for r, cond in enumerate(("before", "after")):
    XY = _2d(E[cond]["fused"])
    axes[r, 0].scatter(XY[:, 0], XY[:, 1], c=cls, cmap="tab20", s=6, alpha=0.7)
    axes[r, 0].set_title(f"{cond} — fused, by mineral class", fontsize=10)
    for lv, col in ((0, "#1f77b4"), (1, "#d62728")):
        m = y == lv
        axes[r, 1].scatter(XY[m, 0], XY[m, 1], c=col, s=6, alpha=0.6,
                           label=str(np.unique(lote)[::-1][lv]))
    axes[r, 1].set_title(f"{cond} — by lote · probe AUC {tbl[cond]['fused']:.2f}", fontsize=10)
    axes[r, 1].legend(markerscale=2, fontsize=8)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
fig.suptitle("Latent geometry before → after constrained training (same grains, same probe)", y=1.0)
fig.tight_layout()

rec = {"date": datetime.date.today().isoformat(), "experiment": "latent-before-after",
       "after_cache": (_afters[-1].name if not AFTER_CACHE else str(AFTER_CACHE)),
       "checkpoint": _acfg.get("checkpoint"), "config": _acfg,
       "n_common": len(common), "base_rate": round(base, 3),
       "probe": tbl, "fused_delta": round(_d, 3), "verdict": verdict}
out = PIPELINE_DIR / "runs" / f"{rec['date']}_latent-before-after.json"
out.write_text(json.dumps(rec, indent=2), encoding="utf-8")
print(f"run log → {out.name}")